# Modeling and feasibility checker

**Group E01** — Project 01, Artificial Intelligence 2026-27. Supporting notebook of the main one.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))   # so Python finds the 'src' folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.model import Problem, State, Stop
from src import informed, local_search, csp

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

prob = Problem.from_csv("../data_tiny")          # development dataset
prob_small = Problem.from_csv("../data_small")   # larger dataset, used to check that the code scales

# 3. Modeling

## 3.1 Decisions used in the implementation

Several points of the brief were left open. The values below come from `rules.csv` when it defines them; the remaining ones are **our own implementation decisions** (marked *ours*) and must be confirmed with the lecturer.

| Topic | Decision | Source |
|---|---|---|
| Time | Minutes since `start_date` 00:00 | ours |
| Waiting | A vehicle that arrives before a window opens waits | ours |
| Time windows | The *service start* must be inside the window | ours |
| Service time | Once per stop (not per item); see `service_min` below | `rules.csv` |
| Warehouse handling | Unloading and loading take `handling_min` each | `rules.csv` |
| Warehouse dwell | An item stays at most `max_dwell_min` in the warehouse | `rules.csv` |
| Flow | producer > warehouse > delivery location | `rules.csv` |
| Producer visit | One visit per producer and day, inside the intersection of the windows of its items | ours |
| End of the route | Every vehicle returns to the collection center | ours |
| Trips | Per vehicle and day: deliver stored orders, collect, deliver the new orders | ours |
| Refrigeration | Stored in the data, but there is no product rule, so it is a soft preference in the CSP only | ours |

In [2]:
print("service_min      :", prob.service_min)
print("handling_min     :", prob.handling_min)
print("max_dwell_min    :", prob.max_dwell_min)
print("depot location_id:", prob.depot_loc)

service_min      : 10.0
handling_min     : 15.0
max_dwell_min    : 1440.0
depot location_id: 4


## 3.2 State and cost

A **state** is a `State(routes)`: for every vehicle a list of `Stop(location_id, action, items)` with the actions `pickup`, `unload`, `load`, `deliver` and `return`. The **cost** is the total distance in km (`Problem.calculate_cost`). `Problem.plan_to_state` turns a plan (visiting order per vehicle and day) into a state.

## 3.3 Feasibility checker: tests with hand-made routes

The checker (`is_feasible`, `check_time`, `is_feasible_full`) must reject wrong routes, not only accept good ones.

In [3]:
import copy

df_model, states_model, plans_model = informed.run_all(prob)
good_plan = plans_model[("astar", "h1")]
good = prob.plan_to_state(good_plan)

# Test 1: the plan found by A* is feasible
print("Test 1 (A* plan)             :", prob.is_feasible_full(good))
assert prob.is_feasible_full(good)

# Test 2: an item is delivered without being picked up (flow violation)
no_flow = State({1: [Stop(prob.depot_loc, "load", (1,)), Stop(7, "deliver", (1,)), Stop(prob.depot_loc, "return", ())]})
print("Test 2 (no pickup)           :", prob.is_feasible(no_flow, check_complete=False))
assert not prob.is_feasible(no_flow, check_complete=False)

# Test 3: all the work of vehicle 2 on vehicle 1 (150 kg) breaks the capacity
overloaded = copy.deepcopy(good_plan)
for day in overloaded:
    for kind in ("collect", "deliver"):
        overloaded[day][kind][1] = overloaded[day][kind][1] + overloaded[day][kind][2]
        overloaded[day][kind][2] = []
print("Test 3 (overloaded vehicle 1):", prob.is_feasible(prob.plan_to_state(overloaded), check_capacity=True))
assert not prob.is_feasible(prob.plan_to_state(overloaded), check_capacity=True)

# Test 4: reversing the delivery order of a vehicle makes it arrive after a window closes
late = copy.deepcopy(good_plan)
day = sorted(late)[-1]
late[day]["deliver"][1] = late[day]["deliver"][1][::-1]
state_late = prob.plan_to_state(late)
print("Test 4 (reversed deliveries) :", prob.check_time(state_late))
print("   violation:", prob.simulate(state_late)[1][0])
assert not prob.check_time(state_late)

Test 1 (A* plan)             : True
Test 2 (no pickup)           : False
Test 3 (overloaded vehicle 1): False
Test 4 (reversed deliveries) : False
   violation: vehicle 1, stop 9 (deliver at 8): starts 31 10:28 after the window closes 31 10:15


In [4]:
# Readable schedule of the A* solution (day 30 = first day, 31 = second day)
prob.route_table(good).head(15)

,vehicle,stop,action,location,arrival,service_start,window,items
0,1,1,pickup,3,30 00:38,30 08:00,30 08:00 - 30 09:45,"[6, 11, 22, 24, 26, 29, 35]"
1,1,2,unload,4,30 08:48,30 08:48,,"[6, 11, 22, 24, 26, 29, 35]"
2,2,1,pickup,1,30 00:40,30 09:30,30 09:30 - 30 09:45,"[5, 13, 25, 31, 32, 34, 37, 40]"
3,2,2,pickup,2,30 10:11,30 10:11,30 09:00 - 30 10:30,"[12, 23, 27, 28, 30, 33, 36, 38, 39]"
4,2,3,unload,4,30 11:11,30 11:11,,"[5, 13, 25, 31, 32, 34, 37, 40, 12, 23, 27, 28..."
5,1,3,load,4,30 09:03,30 11:26,,"[36, 37, 38, 39, 40, 32, 33, 34, 35]"
6,1,4,deliver,6,30 12:29,30 12:29,30 12:15 - 30 14:45,"[36, 37, 38, 39, 40]"
7,1,5,deliver,5,30 13:51,30 13:51,30 10:15 - 30 14:00,"[32, 33, 34, 35]"
8,1,6,return,4,30 14:26,30 14:26,,[]
9,2,4,load,4,30 11:26,30 11:26,,"[30, 31, 11, 12, 13]"
